<a href="https://colab.research.google.com/github/Eashkumar9112003/DNN_Final_template/blob/main/Experiment_notebook_35057938.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

from bs4 import BeautifulSoup
import re
import matplotlib.pyplot as plt
import numpy as np

import random
from PIL import Image

from google.colab import drive
import os

from datasets import load_dataset

from datasets.fingerprint import random
from torch.utils.data import Dataset, DataLoader, random_split
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import torchvision.transforms.functional as FT

from torchvision import transforms, models
from transformers import AutoTokenizer, DistilBertModel, DistilBertTokenizer


import gc

import textwrap


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
# @title Setting up google drive to save checkpoints

# This will prompt you to authorize Google Drive access
drive.mount('/content/gdrive')

def save_checkpoint_to_drive(model, optimizer, epoch, loss, filename="autoencoder_checkpoint.pth"):
    """
    Saves the checkpoint directly to a specified folder in your mounted Google Drive.
    """
    # 1. Define the full Google Drive path
    # 'DL_Checkpoints' is the folder you want to save to inside your Drive
    drive_folder = '/content/gdrive/MyDrive/DL_Checkpoints'

    # Ensure the directory exists before attempting to save
    os.makedirs(drive_folder, exist_ok=True)

    # 2. Combine the folder and the filename
    full_path = os.path.join(drive_folder, filename)

    # 3. Create the checkpoint dictionary
    checkpoint = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'loss': loss,
    }

    # 4. Save the dictionary to the Google Drive path
    torch.save(checkpoint, full_path)
    print(f"Checkpoint saved to Google Drive: {full_path} at epoch {epoch}")


def load_checkpoint_from_drive(model, optimizer=None, filename="autoencoder_checkpoint.pth"):
    """
    Loads a checkpoint from your Google Drive folder into the model and optimizer (if provided).
    """
    # Define the same Google Drive folder path
    drive_folder = '/content/gdrive/MyDrive/DL_Checkpoints'
    full_path = os.path.join(drive_folder, filename)

    # Check if the checkpoint file exists
    if not os.path.exists(full_path):
        raise FileNotFoundError(f"Checkpoint file not found: {full_path}")

    # Load the checkpoint
    checkpoint = torch.load(full_path, map_location=torch.device('cpu'))  # use cuda if available

    # Restore model state
    model.load_state_dict(checkpoint['model_state_dict'])

    # Restore optimizer state (if provided)
    if optimizer is not None:
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])

    # Extract metadata
    epoch = checkpoint.get('epoch', 0)
    loss = checkpoint.get('loss', None)

    print(f"Checkpoint loaded from: {full_path} (epoch {epoch})")

    return model, optimizer, epoch, loss


In [ ]:
# @title Functions to load images and process data


# This function just extracts the tags from the text, don't get distracted by it.
# I changed this function a bit to fix some bugs
def parse_gdi_text(text):
    """Parse GDI formatted text into structured data"""
    soup = BeautifulSoup(text, 'html.parser')
    images = []

    for gdi in soup.find_all('gdi'):
        # Debug: print what BeautifulSoup sees

        # Method 1: Try to get image attribute directly
        image_id = None
        if gdi.attrs:
            # Check for attributes like 'image1', 'image2', etc.
            for attr_name, attr_value in gdi.attrs.items():
                if 'image' in attr_name.lower():
                    image_id = attr_name.replace('image', '')
                    break

        # Method 2: Extract from the tag string using regex
        if not image_id:
            tag_str = str(gdi)
            match = re.search(r'<gdi\s+image(\d+)', tag_str)
            if match:
                image_id = match.group(1)

        # Method 3: Fallback - use sequential numbering
        if not image_id:
            image_id = str(len(images) + 1)

        content = gdi.get_text().strip()

        # Extract tagged elements using BeautifulSoup directly
        objects = [obj.get_text().strip() for obj in gdi.find_all('gdo')]
        actions = [act.get_text().strip() for act in gdi.find_all('gda')]
        locations = [loc.get_text().strip() for loc in gdi.find_all('gdl')]

        images.append({
            'image_id': image_id,
            'description': content,
            'objects': objects,
            'actions': actions,
            'locations': locations,
            'raw_text': str(gdi)
        })

    return images

# This is an utility function to show images.
# Why do we need to do all this?
def show_image(ax, image, de_normalize = False, img_mean = None, img_std = None):
  """
  De-normalize the image (if necessary) and show image
  """
  if de_normalize:
    new_mean = -img_mean/img_std
    new_std = 1/img_std

    image = transforms.Normalize(
        mean=new_mean,
        std=new_std
    )(image)
  ax.imshow(image.permute(1, 2, 0))



In [ ]:
# @title Loading the dataset
train_dataset = load_dataset("daniel3303/StoryReasoning", split="train")
test_dataset = load_dataset("daniel3303/StoryReasoning", split="test")

In [ ]:
raw_dataset = load_dataset("daniel3303/StoryReasoning", split="train")

limited_samples = []

for item in raw_dataset:

    images = item["images"]
    story = item["story"]

    if len(images) >= 5:

        images = images[:5]
        story = story[:5]

        for idx in range(5):

            limited_samples.append({
                "image": images[idx],
                "text": story[idx],
                "label": idx + 1
            })

    if len(limited_samples) >= 1500:
        break

print("Total limited samples:", len(limited_samples))


sample_indices = random.sample(
    range(len(limited_samples)),
    5
)

plt.figure(figsize=(18,6))

for plot_idx, sample_idx in enumerate(sample_indices):

    sample = limited_samples[sample_idx]

    plt.subplot(1,5,plot_idx+1)

    plt.imshow(sample["image"])

    plt.title(
        f"Position: {sample['label']}\n{sample['text'][:40]}...",
        fontsize=8
    )

    plt.axis("off")

plt.suptitle(
    "Sample Story Images with Text Descriptions",
    fontsize=14
)



In [ ]:
sample = raw_dataset[random.randint(0, len(raw_dataset)-1)]

images = sample['images']
story_data = sample['story']


if isinstance(story_data, list):

    story_text = " ".join([str(x) for x in story_data])

else:
    story_text = str(story_data)

clean_text = re.sub(r'<.*?>', '', story_text)

clean_text = clean_text.replace("\n", " ")


wrapped_text = "\n".join(
    textwrap.wrap(clean_text[:1200], width=120)
)


plt.figure(figsize=(20, 12))

num_images = min(5, len(images))

for i in range(num_images):

    plt.subplot(3, 3, i+1)

    plt.imshow(images[i])

    plt.axis('off')

    plt.title(
        f"Sequence position {i+1}",
        fontsize=12,
        color="red",
        fontweight="bold"
    )


plt.subplot(3,1,3)

plt.axis('off')

plt.text(
    0,
    0.9,
    wrapped_text,
    fontsize=11,
    verticalalignment='top'
)

plt.suptitle(
    "Story Dataset Sample with Narrative Text",
    fontsize=18,
    fontweight="bold"
)

plt.show()

print("Figure saved successfully:")

In [ ]:
print(type(raw_dataset))

print("\nDataset Length:")
print(len(raw_dataset))

print("\nKeys:")
print(raw_dataset[0].keys())

In [ ]:
sample = raw_dataset[0]

print("Number of Images:")
print(len(sample["images"]))

print("\nType of story:")
print(type(sample["story"]))

print("\nStory Content:")
print(sample["story"])

In [ ]:
sample = raw_dataset[0]

images = sample["images"]
stories = sample["story"]

print("Images:", len(images))

if isinstance(stories, list):
    print("Story sentences:", len(stories))
else:
    print("Story is stored as single text block")

In [ ]:

sample = raw_dataset[random.randint(0, len(raw_dataset)-1)]

images = sample["images"]
story_text = sample["story"]


# Extract stories linked to each image


story_blocks = re.findall(
    r"<gdi image\d+>(.*?)</gdi>",
    story_text,
    re.DOTALL
)


cleaned_stories = []

for block in story_blocks:

    clean = re.sub(r"<.*?>", "", block)

    clean = clean.replace("\n", " ")

    clean = clean.strip()

    cleaned_stories.append(clean)


fig, axes = plt.subplots(2, 3, figsize=(22, 14))

axes = axes.flatten()

num_samples = min(6, len(images), len(cleaned_stories))

for i in range(num_samples):

    axes[i].imshow(images[i])

    axes[i].set_title(
        f"Label: {i+1}",
        fontsize=15,
        color="red",
        fontweight="bold"
    )

    wrapped_story = textwrap.fill(
        cleaned_stories[i][:350],
        width=38
    )

    axes[i].text(
        0.5,
        -0.18,
        wrapped_story,
        fontsize=8,
        ha='center',
        va='top',
        transform=axes[i].transAxes
    )

    axes[i].axis("off")

plt.suptitle(
    "Story Dataset Samples with Image-wise Narrative Alignment",
    fontsize=20,
    fontweight="bold"
)

plt.subplots_adjust(
    hspace=1.0,
    wspace=0.3
)

plt.show()

print("Saved successfully:")

In [ ]:

from PIL import Image

processed_samples = []

for item in raw_dataset:

    images = item["images"]

    story_text = item["story"]

    story_blocks = re.findall(
        r"<gdi image\d+>(.*?)</gdi>",
        story_text,
        re.DOTALL
    )

    # Clean text
    cleaned_stories = []

    for block in story_blocks:

        clean = re.sub(r"<.*?>", "", block)

        clean = clean.replace("\n", " ")

        clean = clean.strip()

        cleaned_stories.append(clean)

    # Align image + label
    usable = min(5, len(images), len(cleaned_stories))

    for idx in range(usable):

        processed_samples.append({
            "image": images[idx],
            "story": cleaned_stories[idx],
            "label": idx
        })


# Limit samples


processed_samples = processed_samples[:1500]

print("Total processed samples:", len(processed_samples))

print("\nExample sample:")

print("Label:", processed_samples[0]["label"])

print("Story:", processed_samples[0]["story"][:200])

In [ ]:
#  2: Robust Dataset
class StoryDataset(Dataset):
    def __init__(self, hf_dataset, tokenizer, transform, seq_len=4):
        self.data = hf_dataset
        self.tokenizer = tokenizer
        self.transform = transform
        self.seq_len = seq_len
        self.needed_len = seq_len + 1

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]

        images_list = item['images']
        texts_list = item['story']

        if isinstance(texts_list, str):
            texts_list = [texts_list]

        # Pad Images
        if len(images_list) < self.needed_len:
            diff = self.needed_len - len(images_list)
            images_list = images_list + [images_list[-1]] * diff

        # Pad Texts
        if len(texts_list) < self.needed_len:
            diff_t = self.needed_len - len(texts_list)
            if len(texts_list) == 0:
                texts_list = ["empty"] * self.needed_len
            else:
                texts_list = texts_list + [texts_list[-1]] * diff_t

        # Slice last 5
        imgs = images_list[-self.needed_len:]
        txts = texts_list[-self.needed_len:]

        # Transform Images
        processed_imgs = []
        for img in imgs:
            if img.mode != 'RGB':
                img = img.convert('RGB')
            processed_imgs.append(self.transform(img))

        img_tensor = torch.stack(processed_imgs)

        # Tokenize Text
        encoded_text = self.tokenizer(
            txts,
            padding='max_length',
            truncation=True,
            max_length=32,
            return_tensors='pt'
        )

        # Split 4 inputs, 1 target
        input_images = img_tensor[:-1]
        input_ids = encoded_text['input_ids'][:-1]
        attention_mask = encoded_text['attention_mask'][:-1]

        target_image = img_tensor[-1]
        target_ids = encoded_text['input_ids'][-1]

        return {
            "input_images": input_images,
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "target_image": target_image,
            "target_ids": target_ids
        }


print("Downloading Dataset...")
raw_dataset = load_dataset("daniel3303/StoryReasoning", split="train[:300]")

# Image Transform
img_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")

# Define CONFIG dictionary
CONFIG = {
    "seq_len": 4,
    "batch_size": 32,
    "embed_dim": 256,
    "hidden_dim": 512,
    "vocab_size": tokenizer.vocab_size, # Get from tokenizer
    "learning_rate": 0.001,
    "epochs": 10,
    "lambda_contrastive": 0.1
}

train_dataset = StoryDataset(
    raw_dataset,
    tokenizer,
    img_transform,
    seq_len=CONFIG["seq_len"]
)

train_loader = DataLoader(
    train_dataset,
    batch_size=CONFIG["batch_size"],
    shuffle=True,
    drop_last=True
)

print(f"Data Loaded. Dataset size: {len(train_dataset)}")

# Debug sample
sample = train_dataset[0]
print(f"Sample Input Shape: {sample['input_ids'].shape}")

if sample['input_ids'].shape[0] == 4:
    print("SUCCESS: Input shape is correct [4, 32].")
else:
    print("ERROR: Shape is not correct.")


In [ ]:
story_lengths = [len(item["images"]) for item in raw_dataset]

plt.figure(figsize=(8,5))

plt.hist(story_lengths, bins=10)

plt.xlabel("Original Story Length (images)")

plt.ylabel("Count")

plt.title("Distribution of Story Lengths (Before Padding)")

plt.show()

print("Plot saved at:")


In [ ]:

sample = train_dataset[0]

plt.figure(figsize=(12, 3))

for i in range(5):

    if i < 4:

        img = sample["input_images"][i].permute(1, 2, 0)

        title = f"Input {i+1}"

    else:

        img = sample["target_image"].permute(1, 2, 0)

        title = "Target"

    plt.subplot(1, 5, i+1)

    plt.imshow(img)

    plt.axis("off")

    plt.title(title)

plt.suptitle("Story Sequence (4 Inputs + 1 Target)")

plt.show()

print("Output saved successfully at:")


In [ ]:

batch = next(iter(train_loader))

img_shape = batch["input_images"].shape

text_shape = batch["input_ids"].shape

plt.figure(figsize=(6,4))

plt.bar(
    ["Images Batch", "Text Batch"],
    [img_shape[0], text_shape[0]]
)

plt.ylabel("Batch Size")

plt.title("Batch Size Consistency Check")

plt.show()

print("Image batch shape:", img_shape)

print("Text batch shape:", text_shape)

print("\nOutput saved successfully at:")


In [ ]:

class VisualEncoder(nn.Module):

    def __init__(self, embed_dim):

        super().__init__()

        resnet = models.resnet50(pretrained=True)

        self.backbone = nn.Sequential(
            *list(resnet.children())[:-1]
        )

        self.fc = nn.Linear(
            resnet.fc.in_features,
            embed_dim
        )

    def forward(self, x):

        b, s, c, h, w = x.shape

        x = x.view(b * s, c, h, w)

        features = self.backbone(x)

        features = features.view(features.size(0), -1)

        features = self.fc(features)

        return features.view(b, s, -1)


class TextEncoder(nn.Module):

    def __init__(self, embed_dim):

        super().__init__()

        self.bert = DistilBertModel.from_pretrained(
            "distilbert-base-uncased"
        )

        self.fc = nn.Linear(768, embed_dim)

    def forward(self, input_ids, attention_mask):

        b, s, l = input_ids.shape

        flat_ids = input_ids.view(b * s, l)

        flat_mask = attention_mask.view(b * s, l)

        output = self.bert(
            flat_ids,
            attention_mask=flat_mask
        )

        cls_token = output.last_hidden_state[:, 0, :]

        features = self.fc(cls_token)

        return features.view(b, s, -1)


visual_encoder = VisualEncoder(embed_dim=256)

text_encoder = TextEncoder(embed_dim=256)

print("Encoder architecture saved successfully at:")


In [ ]:
def contrastive_loss(
    text_embeds,
    visual_embeds,
    temperature=0.07
):

    assert text_embeds.shape == visual_embeds.shape, \
        "Shape mismatch: text and image embeds must align"

    # Normalize embeddings
    t = F.normalize(text_embeds, p=2, dim=1)

    v = F.normalize(visual_embeds, p=2, dim=1)

    # Similarity matrix
    sim_matrix = torch.matmul(t, v.T) / temperature

    # Labels
    labels = torch.arange(
        sim_matrix.size(0),
        device=sim_matrix.device
    )

    # Symmetric CE loss
    loss_forward = F.cross_entropy(sim_matrix, labels)

    loss_backward = F.cross_entropy(sim_matrix.T, labels)

    return (loss_forward + loss_backward) * 0.5

class StoryReasoningModel(nn.Module):

    def __init__(self, config):

        super().__init__()

        self.visual_enc = VisualEncoder(
            config["embed_dim"]
        )

        self.text_enc = TextEncoder(
            config["embed_dim"]
        )


        # Fusion Layer


        self.fusion_fc = nn.Linear(
            config["embed_dim"] * 2,
            config["hidden_dim"]
        )


        # Temporal LSTM


        self.temporal_lstm = nn.LSTM(
            input_size=config["hidden_dim"],
            hidden_size=config["hidden_dim"],
            num_layers=1,
            batch_first=True
        )


        # Text Decoder


        self.text_embed = nn.Embedding(
            config["vocab_size"],
            config["embed_dim"]
        )

        self.text_decoder_gru = nn.GRU(
            config["embed_dim"],
            config["hidden_dim"],
            batch_first=True
        )

        self.text_out_fc = nn.Linear(
            config["hidden_dim"],
            config["vocab_size"]
        )


        # Image Decoder


        self.image_decoder = nn.Sequential(

            nn.Linear(
                config["hidden_dim"],
                256 * 7 * 7
            ),

            nn.ReLU(),

            nn.Unflatten(
                dim=1,
                unflattened_size=(256, 7, 7)
            ),

            nn.ConvTranspose2d(256,128,4,2,1),
            nn.ReLU(),

            nn.ConvTranspose2d(128,64,4,2,1),
            nn.ReLU(),

            nn.ConvTranspose2d(64,32,4,2,1),
            nn.ReLU(),

            nn.ConvTranspose2d(32,16,4,2,1),
            nn.ReLU(),

            nn.ConvTranspose2d(16,3,4,2,1),

            nn.Sigmoid()
        )


    # FORWARD PASS


    def forward(
        self,
        images,
        input_ids,
        mask,
        target_ids=None
    ):

        # Encode
        v_emb = self.visual_enc(images)

        t_emb = self.text_enc(
            input_ids,
            mask
        )

        B, S, E = v_emb.size()

        # Contrastive alignment
        align_loss = contrastive_loss(
            t_emb.reshape(B*S, E),
            v_emb.reshape(B*S, E)
        )

        # Fusion
        fused = torch.cat(
            (v_emb, t_emb),
            dim=-1
        )

        fused = F.relu(
            self.fusion_fc(fused)
        )

        # Temporal reasoning
        _, (h_last, _) = self.temporal_lstm(fused)

        context_vector = h_last[-1]

        # Image decoder
        pred_img = self.image_decoder(context_vector)

        # Text decoder
        if target_ids is not None:

            tgt_embed = self.text_embed(target_ids)

            decoder_init = context_vector.unsqueeze(0)

            dec_out, _ = self.text_decoder_gru(
                tgt_embed,
                decoder_init
            )

            text_logits = self.text_out_fc(dec_out)

        else:

            text_logits = None

        return pred_img, text_logits, align_loss


# INITIALIZE MODEL


model = StoryReasoningModel(CONFIG).to(device)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=CONFIG["learning_rate"]
)

print("Model initialized successfully.")

print("\nModel architecture saved successfully at:")


In [ ]:
import os
import gc
import json
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

BASE_PATH = "/content/gdrive/MyDrive/DL_Checkpoints"
OUTPUT_PATH = f"{BASE_PATH}/results/outputs"
os.makedirs(OUTPUT_PATH, exist_ok=True)

TRAIN_LOG_FILE = f"{OUTPUT_PATH}/optimized_training_log.txt"

LOSS_PLOT_FILE = f"{OUTPUT_PATH}/optimized_training_loss.png"

SUMMARY_FILE = f"{OUTPUT_PATH}/training_summary.txt"

for p in model.visual_enc.backbone.parameters():

    p.requires_grad = False

for p in model.text_enc.bert.parameters():

    p.requires_grad = False

print("Backbone freezing complete: ResNet + DistilBERT locked.")

train_loader = DataLoader(
    train_dataset,
    batch_size=2,
    shuffle=True,
    drop_last=True
)

print("Adjusted DataLoader: batch_size = 2")


def train_one_epoch_optimized(
    model,
    loader,
    optimizer
):

    model.train()

    epoch_loss = 0.0

    print(f"→ Training step initiated on {len(loader)} mini-batches")

    for batch_idx, batch in enumerate(loader):


        imgs = batch["input_images"].to(device)

        ids = batch["input_ids"].to(device)

        mask = batch["attention_mask"].to(device)

        tgt_img = batch["target_image"].to(device)

        tgt_ids = batch["target_ids"].to(device)

        optimizer.zero_grad()

        pred_img, text_logits, align_loss = model(
            imgs,
            ids,
            mask,
            tgt_ids
        )


        img_loss = F.mse_loss(
            pred_img,
            tgt_img
        )


        vocab_dim = CONFIG["vocab_size"]

        text_loss = F.cross_entropy(
            text_logits.reshape(-1, vocab_dim),
            tgt_ids.reshape(-1),
            ignore_index=tokenizer.pad_token_id
        )


        combined = (
            img_loss +
            text_loss +
            CONFIG["lambda_contrastive"] * align_loss
        )

        combined.backward()

        optimizer.step()

        epoch_loss += combined.item()


        del imgs, ids, mask
        del tgt_img, tgt_ids
        del pred_img, text_logits
        del img_loss, text_loss
        del align_loss, combined

        if batch_idx % 10 == 0:

            torch.cuda.empty_cache()

            gc.collect()

    return epoch_loss / len(loader)

print("\n→ Starting model training loop...\n")

epoch_losses = []

try:

    for epoch in range(10):

        loss_val = train_one_epoch_optimized(
            model,
            train_loader,
            optimizer
        )

        epoch_losses.append(loss_val)

        log_text = (
            f"Epoch [{epoch+1}/{CONFIG['epochs']}]\n"
            f"Training Loss: {loss_val:.4f}\n"
            f"{"-"*50}\n"
        )


        print(log_text)

except RuntimeError as error_msg:

    if "out of memory" in str(error_msg).lower():

        print("\nOOM detected → try seq_len = 2 inside CONFIG.")

        torch.cuda.empty_cache()

    else:

        raise error_msg

plt.figure(figsize=(7,5))

plt.plot(
    range(1, len(epoch_losses)+1),
    epoch_losses,
    marker='o'
)

plt.xlabel("Epoch")

plt.ylabel("Training Loss")

plt.title("Optimized Training Loss Curve")

plt.grid(True)

plt.show()


In [ ]:
!pip install evaluate
!pip install rouge_score

import os
import json
import torch
import evaluate

BASE_PATH = "/content/gdrive/MyDrive/DL_Checkpoints"


OUTPUT_PATH = f"{BASE_PATH}/results/outputs"

os.makedirs(OUTPUT_PATH, exist_ok=True)

EVAL_TEXT_FILE = f"{OUTPUT_PATH}/evaluation_results.txt"

PREDICTION_FILE = f"{OUTPUT_PATH}/sample_predictions.txt"

METRICS_JSON_FILE = f"{OUTPUT_PATH}/evaluation_metrics.json"


bleu = evaluate.load("bleu")

meteor = evaluate.load("meteor")

rouge = evaluate.load("rouge")

def evaluate_model(model, loader):

    model.eval()

    refs = []

    preds = []

    sample_outputs = []

    print("Generating predictions for evaluation...")

    with torch.no_grad():

        for i, batch in enumerate(loader):

            if i >= 5:
                break

            imgs = batch['input_images'].to(device)

            inp_ids = batch['input_ids'].to(device)

            mask = batch['attention_mask'].to(device)

            tgt_ids = batch['target_ids']


            _, text_logits, _ = model(
                imgs,
                inp_ids,
                mask,
                batch['target_ids'].to(device)
            )


            pred_ids = torch.argmax(
                text_logits,
                dim=-1
            )

            decoded_preds = tokenizer.batch_decode(
                pred_ids,
                skip_special_tokens=True
            )

            decoded_refs = tokenizer.batch_decode(
                tgt_ids,
                skip_special_tokens=True
            )

            preds.extend(decoded_preds)

            refs.extend(decoded_refs)


            for p, r in zip(decoded_preds, decoded_refs):

                sample_outputs.append(
                    f"\nREFERENCE:\n{r}\n\nPREDICTION:\n{p}\n"
                    + "="*80
                )


    b_score = bleu.compute(
        predictions=preds,
        references=[[r] for r in refs]
    )

    r_score = rouge.compute(
        predictions=preds,
        references=refs
    )

    m_score = meteor.compute(
        predictions=preds,
        references=refs
    )


    result_text = f"""

BLEU Score     : {b_score['bleu']:.4f}

ROUGE-L Score  : {r_score['rougeL']:.4f}

METEOR Score   : {m_score['meteor']:.4f}

"""

    print(result_text)

    if b_score['bleu'] > 0.1 and r_score['rougeL'] < 0.1:

        analysis_text = (
            "\n[Analysis] High BLEU but Low ROUGE detected.\n"
            "The Contrastive Loss (Innovation) was active "
            "to mitigate this.\n"
        )

        print(analysis_text)

        result_text += analysis_text


    with open(EVAL_TEXT_FILE, "w") as f:

        f.write(result_text)


    with open(PREDICTION_FILE, "w") as f:

        for item in sample_outputs:

            f.write(item)

    metrics_dict = {

        "BLEU": float(b_score['bleu']),

        "ROUGE-L": float(r_score['rougeL']),

        "METEOR": float(m_score['meteor'])
    }

    with open(METRICS_JSON_FILE, "w") as f:

        json.dump(metrics_dict, f, indent=4)


    print("Evaluation results saved successfully.\n")

    print("Saved Files:")

    print(PREDICTION_FILE)

    print(METRICS_JSON_FILE)
